# 02 — Exploratory Data Analysis (EDA)
**Project:** CustomerPulse AI — E-commerce Customer Intelligence & Churn Prediction System

### Objective
Analyze store-wide and customer-level purchasing patterns across `531,211` cleaned store transactions (`399,656` identified customer transactions) from `2010-12-01` to `2011-12-09`. Every visual analysis below documents:
1. **What it measures**
2. **Empirical chart & table output**
3. **Interpretation of actual results**
4. **Business implication**
5. **Dataset limitations**

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import EDA_REPORT_JSON_PATH, FIGURES_DIR

with open(EDA_REPORT_JSON_PATH, "r", encoding="utf-8") as f:
    eda_data = json.load(f)

pd.DataFrame([eda_data["kpis"]]).T.rename(columns={0: "Verified Value"})

## Chart 1: Monthly Gross Qualifying Sales vs. Net Revenue Trend
1. **What it measures:** Monthly gross qualifying merchandise sales (`Quantity > 0`) vs. net revenue after subtracting cancelled/returned invoices (`InvoiceNo` starting with `'C'`).

In [ ]:
display(Image(filename=str(FIGURES_DIR / "01_monthly_revenue_trend.png")))
pd.DataFrame(eda_data["monthly_revenue_trend"])

- **Interpretation:** Store revenue grows strongly during Q4 holiday wholesale stocking—rising from ~£669k gross in August 2011 to £1.03M in September, £1.11M in October, and peaking at **£1.48M gross (£1.42M net) in November 2011**.
- **Business Implication:** Inventory procurement and customer retention campaigns must be executed in **July–August** ahead of the September–November Q4 wholesale surge. Losing a wholesale buyer prior to September costs disproportionately more annual revenue.
- **Limitations:**
  1. `2011-12` only contains **9 days of data** (up to `2011-12-09`), which makes December 2011 look artificially lower than November 2011.
  2. In `2011-01` and `2011-12`, single massive orders (`74,215` units of *Medium Ceramic Top Storage Jar* in Jan 2011 for £77,183.60, and `80,995` units of *Paper Craft, Little Birdie* in Dec 2011 for £168,469.60) were placed and cancelled within minutes, inflating both Gross Revenue and Cancellations in those two months.

## Chart 2: Top 10 Products by Gross Revenue & Units Sold
1. **What it measures:** The top-performing physical SKUs ranked by gross qualifying sales (£) and total units sold.

In [ ]:
display(Image(filename=str(FIGURES_DIR / "02_top_products_revenue_and_quantity.png")))
pd.DataFrame(eda_data["top_10_highest_revenue_products"])

- **Interpretation:** Consistent multi-order staples like `22423` (*Regency Cakestand 3 Tier*, £202,437 across 1,988 orders), `85123A` (*White Hanging Heart T-Light Holder*, £104,462 across 2,256 orders), and `47566` (*Party Bunting*, £99,437 across 1,686 orders) drive recurring revenue.
- **Business Implication:** High-frequency anchor products (*Regency Cakestand*, *White Hanging Heart T-Light Holder*, *Jumbo Bag Red Retrospot*) act as basket builders and ideal cross-sell hooks for reactivation emails.
- **Limitations:** Single-order anomalies (`23843` *Paper Craft, Little Birdie* at £168,469 from 1 order that was immediately cancelled, and `23166` *Medium Ceramic Top Storage Jar* at £81,700 where £77,183 was cancelled) appear in Gross Sales rankings unless cross-checked against net sales and cancellation tables.

## Chart 3: Geographic Revenue Concentration
1. **What it measures:** Gross qualifying revenue and order volume across countries.

In [ ]:
display(Image(filename=str(FIGURES_DIR / "03_top_international_countries.png")))
pd.DataFrame(eda_data["top_10_countries_by_revenue"])

- **Interpretation:** The **United Kingdom** accounts for **84.56%** of store-wide gross revenue (£8.82M), followed by top European wholesale hubs: **Netherlands** (£283.9k), **EIRE / Ireland** (£276.6k), **Germany** (£205.6k), and **France** (£184.9k).
- **Business Implication:** Because non-UK European buyers place larger wholesale orders (higher Average Order Value) with higher shipping friction, country/region (`is_uk`) should be tested as a feature in our customer segmentation and churn models.
- **Limitations:** Because non-UK customer counts per individual country are small, grouping international countries into `UK` vs. `International` prevents high-cardinality overfitting during machine learning training.

## Chart 4: Customer Spending Skewness & Repeat Purchase Frequency
1. **What it measures:** The distribution of cumulative spend and order count across the `4,334` active identified customers.

In [ ]:
display(Image(filename=str(FIGURES_DIR / "04_customer_spending_and_repeat_orders.png")))
pd.DataFrame([eda_data["customer_spend_distribution"]])

- **Interpretation:**
  - **Extreme Right Skewness (Pareto Effect):** Median customer spend is **£668.43**, whereas mean spend is **£2,015.97**. The **top 10% of customers generate 61.2% of revenue**, and the **top 20% generate 74.6% of revenue**.
  - **Repeat Purchase Rate:** **65.51%** (`2,839` customers) placed 2 or more orders, while **34.49%** (`1,495` customers) placed only 1 order.
- **Business Implication:** Converting a first-time buyer into a second-order buyer and preventing churn among top-decile wholesale accounts are the two highest-ROI levers for the business.
- **Limitations:** Guest checkouts (`24.93%` of rows) lack a `CustomerID` and therefore cannot be tracked longitudinally at the individual customer level.

## Chart 5: Day-of-Week and Time-of-Day Seasonality
1. **What it measures:** Operational order flow across days of the week and hours of the day.

In [ ]:
display(Image(filename=str(FIGURES_DIR / "05_seasonality_day_and_hour.png")))

- **Interpretation:** Order volume peaks on **Thursdays** (with Thursday evening extending until 8:00 PM) and between **10:00 AM and 3:00 PM** (peaking at **12:00 PM noon**). Notice there are **zero orders on Saturdays** because the UK wholesale retailer did not process invoices on Saturdays.
- **Business Implication:** Automated retention emails, promotional outreach, and database batch refreshes should be scheduled for Tuesday–Thursday mornings (9:00 AM – 11:00 AM UK time) when wholesale buyers are actively placing orders.
- **Limitations:** `InvoiceDate` reflects invoice generation timestamp in the retailer's ERP system rather than web-cart browsing timestamp.